In [1]:
import numpy as np
from scipy.optimize import curve_fit

try:
    from sbpy.photometry import HG1G2
    import astropy.units as u
except ImportError:
    raise ImportError("This script requires 'sbpy' and 'astropy'. Install them via: pip install sbpy astropy")



In [2]:

def sphere_scattering_base(alpha, c=0.1):
    """
    Analytic integration of Lommel-Seeliger (LS) and Lambert (L) 
    scattering over a perfectly spherical asteroid.
    
    Args:
        alpha (array): Phase angles in radians.
        c (float): Lambert weight (default 0.1 for DAMIT).
    Returns:
        array: Base relative scattering shape of the sphere.
    """
    # Prevent division by zero / log(0) at exact opposition
    alpha = np.where(alpha == 0, 1e-8, alpha)
    
    # Analytic Lommel-Seeliger integral over a sphere (normalized to 1 at alpha=0)
    S_LS_rel = 1 - np.sin(alpha/2) * np.tan(alpha/2) * np.log(1 / np.tan(alpha/4))
    
    # Analytic Lambert integral over a sphere (normalized to 1 at alpha=0)
    S_L_rel = (np.sin(alpha) + (np.pi - alpha) * np.cos(alpha)) / np.pi
    
    # Combine using flux ratios: F_LS(0) = pi/3, F_L(0) = pi*c/2
    weight_LS = 2
    weight_L = 3 * c
    return (weight_LS * S_LS_rel + weight_L * S_L_rel) / (weight_LS + weight_L)

def convert_HG1G2_to_adk(G1, G2, c=0.1, alpha_max_deg=30.0):
    """
    Fits the DAMIT a, d, k phase parameters to Muinonen G1, G2 parameters.
    
    Args:
        G1 (float): The G1 parameter.
        G2 (float): The G2 parameter.
        c (float): The Lambert weight assumed in the shape model (usually 0.1).
        alpha_max_deg (float): The maximum phase angle to fit over (in degrees).
        
    Returns:
        tuple: (a, d, k) parameters fit to the curve.
    """
    # Generate phase angles
    alphas_deg = np.linspace(0, alpha_max_deg, 200)
    alphas_rad = np.radians(alphas_deg)
    
    # Use sbpy to evaluate the H, G1, G2 model via its built-in cubic splines
    # We set H=0 magnitudes so that the flux normalizes cleanly to 1.0 at 0 degrees
    model = HG1G2(H=0 * u.mag, G1=G1, G2=G2)
    mags = model(alphas_deg * u.deg)
    
    # Convert absolute magnitude outputs to normalized relative intensity
    target_intensity = 10**(-0.4 * mags.value)
    
    # The mathematical model representing the resolved a,d,k phase law
    def damit_model_intensity(alpha, a, d, k):
        f_alpha = 1 + a * np.exp(-alpha / d) + k * alpha
        f_0 = 1 + a 
        return (f_alpha / f_0) * sphere_scattering_base(alpha, c)

    # Initial guess: standard default values (a, d, k)
    p0 = [0.5, 0.1, -0.5]
    
    # Bounds: a > 0, d > 0, k is a negative slope
    bounds = (
        [0.0, 0.001, -5.0], 
        [5.0, 1.0, 0.0]
    )
    
    # Perform Least Squares Curve Fit
    popt, _ = curve_fit(damit_model_intensity, alphas_rad, target_intensity, p0=p0, bounds=bounds)
    
    a, d, k = popt
    return a, d, k


In [4]:
def bowell_hg_intensity(alpha, G):
    """
    Computes the relative intensity of the Bowell et al. (1989) H, G model.
    Note: Absolute magnitude (H) acts merely as a scaling factor, so we 
    only need G to determine the shape of the relative phase curve.
    
    Args:
        alpha (array): Phase angles in radians.
        G (float): Slope parameter G.
    Returns:
        array: Normalized intensity (1.0 at alpha=0).
    """
    # Standard basis functions for the H, G system
    phi1 = np.exp(-3.332 * np.tan(alpha / 2)**0.63)
    phi2 = np.exp(-1.862 * np.tan(alpha / 2)**1.218)
    return (1 - G) * phi1 + G * phi2

# def sphere_scattering_base(alpha, c=0.1):
#     """
#     Analytic integration of Lommel-Seeliger (LS) and Lambert (L) 
#     scattering over a perfectly spherical asteroid.
    
#     Args:
#         alpha (array): Phase angles in radians.
#         c (float): Lambert weight (default 0.1 for DAMIT).
#     Returns:
#         array: Base relative scattering shape of the sphere.
#     """
#     # Prevent division by zero / log(0) at exact opposition
#     alpha = np.where(alpha == 0, 1e-8, alpha)
    
#     # Analytic Lommel-Seeliger integral over a sphere (normalized to 1 at alpha=0)
#     S_LS_rel = 1 - np.sin(alpha/2) * np.tan(alpha/2) * np.log(1 / np.tan(alpha/4))
    
#     # Analytic Lambert integral over a sphere (normalized to 1 at alpha=0)
#     S_L_rel = (np.sin(alpha) + (np.pi - alpha) * np.cos(alpha)) / np.pi
    
#     # Combine using flux ratios: F_LS(0) = pi/3, F_L(0) = pi*c/2
#     # This weights the combined normalized shape properly.
#     weight_LS = 2
#     weight_L = 3 * c
#     return (weight_LS * S_LS_rel + weight_L * S_L_rel) / (weight_LS + weight_L)

def convert_G_to_adk(G, c=0.1, alpha_max_deg=30.0):
    """
    Fits the DAMIT a, d, k phase parameters to an IAU G parameter.
    
    Args:
        G (float): The Bowell G parameter.
        c (float): The Lambert weight assumed in the shape model (usually 0.1).
        alpha_max_deg (float): The maximum phase angle to fit over (in degrees).
        
    Returns:
        tuple: (a, d, k) parameters fit to the curve.
    """
    # Generate phase angles from 0 to alpha_max
    alphas_deg = np.linspace(0, alpha_max_deg, 200)
    alphas_rad = np.radians(alphas_deg)
    
    # Generate target intensity curve from the H, G model
    target_intensity = bowell_hg_intensity(alphas_rad, G)
    
    # The mathematical model representing the resolved a,d,k phase law applied to a sphere
    def damit_model_intensity(alpha, a, d, k):
        # f(alpha) = 1 + a*exp(-alpha/d) + k*alpha
        f_alpha = 1 + a * np.exp(-alpha / d) + k * alpha
        f_0 = 1 + a # Value at zero phase
        
        # Total modeled disk-integrated intensity
        return (f_alpha / f_0) * sphere_scattering_base(alpha, c)

    # Initial guess: standard default values
    # a: amplitude, d: width (rad), k: slope
    p0 = [0.5, 0.1, -0.5]
    
    # Bounds:
    # a must be positive, d must be positive (to avoid div-by-zero), k is negative slope
    bounds = (
        [0.0, 0.001, -5.0], 
        [5.0, 1.0, 0.0]
    )
    
    # Perform Least Squares Curve Fit
    popt, _ = curve_fit(damit_model_intensity, alphas_rad, target_intensity, p0=p0, bounds=bounds)
    
    a, d, k = popt
    return a, d, k


In [14]:
# Test with typical S-type asteroid parameters
test_G1 = 0.1
test_G2 = 0.1

# DAMIT standard Lambert weight
lambert_weight_c = 0.1

a_fit, d_fit, k_fit = convert_HG1G2_to_adk(test_G1, test_G2, c=lambert_weight_c)

print(f"Given Muinonen G1 = {test_G1}, G2 = {test_G2}:")
print("-" * 35)
print(f"a (Opposition Amplitude) : {a_fit:.4f}")
print(f"d (Opposition Width rad) : {d_fit:.4f}  ({np.degrees(d_fit):.2f}°)")
print(f"k (Linear Slope)         : {k_fit / 180 * np.pi:.4f}° per degree of phase angle")

Given Muinonen G1 = 0.1, G2 = 0.1:
-----------------------------------
a (Opposition Amplitude) : 2.4780
d (Opposition Width rad) : 0.0338  (1.94°)
k (Linear Slope)         : -0.0230° per degree of phase angle


In [11]:
# Test with standard S-type asteroid slope (G ~ 0.25)
test_G = 0.1

# DAMIT standard Lambert weight
lambert_weight_c = 0.1

a_fit, d_fit, k_fit = convert_G_to_adk(test_G, c=lambert_weight_c)

print(f"Given Bowell G = {test_G}:")
print("-" * 30)
print(f"a (Opposition Amplitude) : {a_fit:.4f}")
print(f"d (Opposition Width rad) : {d_fit:.4f}  ({np.degrees(d_fit):.2f}°)")
print(f"k (Linear Slope)   per rad      : {k_fit:.4f}")
print(f"k (Linear Slope) degrees: {(k_fit * np.pi / 180):.4f}° per degree of phase angle")

Given Bowell G = 0.1:
------------------------------
a (Opposition Amplitude) : 0.6139
d (Opposition Width rad) : 0.0601  (3.44°)
k (Linear Slope)   per rad      : -0.9640
k (Linear Slope) degrees: -0.0168° per degree of phase angle
